# Basic Multi Layer Perceptron using Iris Dataset

## a. Lab requirements

The second lab assignment is to use scikit-learn to implement a neural network for the Iris dataset, which is available on the Moodle page.

1. Construct a neural network consisting of:

    • an input layer with 4 nodes

    • at least one hidden layer (with, say, 20 nodes)

    • an output layer with 3 nodes. You may add more hidden layers if you wish, but at least one is required.

2. Prepare the Iris dataset by:

    (a) normalising the input values using max-min normalisation.

    (b) creating a one-hot encoding of the target values.

3. Split the dataset into training, validation, and test sets.

4. Train the neural network on the prepared Iris dataset. Use the sum-of-squares loss
function.

5. During training, after every epoch, print:

    • the sum-of-squares loss on the training set.

6. After training, print the accuracy on the test set.

## b. Imports

In [38]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score

## c. Load Data

In [39]:
#load data locally
cols = ["x1", "x2", "x3", "x4", "target"]
df = pd.read_csv("Iris.csv", sep=";", header=None, names=cols)
df.head()

,x1,x2,x3,x4,target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## d. Data Cleaning

In [40]:
#Checking the data profile
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      150 non-null    float64
 1   x2      150 non-null    float64
 2   x3      150 non-null    float64
 3   x4      150 non-null    float64
 4   target  150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


### Key Takeaways:

1. We have no missing values in any of the columns.

2. The target column is of string type. Need conversion to integer before modeling; One hot encoding has been recommended in the requirements.



In [41]:
#check for duplicates
total_duplicates = df.duplicated().sum()
print(int(total_duplicates))

3


We have three duplicates in our dataset.

In [42]:
df[df.duplicated()]

,x1,x2,x3,x4,target
34,4.9,3.1,1.5,0.1,Iris-setosa
37,4.9,3.1,1.5,0.1,Iris-setosa
142,5.8,2.7,5.1,1.9,Iris-virginica


In [43]:
#dropt the duplicates
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 147 entries, 0 to 149
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      147 non-null    float64
 1   x2      147 non-null    float64
 2   x3      147 non-null    float64
 3   x4      147 non-null    float64
 4   target  147 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.9 KB


I dropped all the duplicates keepiing the first occurances only. This reduced or dataset to 147 from 150 values.

In [44]:
#Checking the summary stats
df.describe()

,x1,x2,x3,x4
count,147.000000,147.000000,147.000000,147.000000
mean,5.856463,3.055782,3.780272,1.208844
std,0.829100,0.437009,1.759111,0.757874
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.400000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [45]:
df.groupby(["target"])["target"].count()

target
Iris-setosa        48
Iris-versicolor    50
Iris-virginica     49
Name: target, dtype: int64

The target column is pretty much balanced.

## e. Split the Raw Data

In [46]:
#Split to features and target
X = df[["x1", "x2", "x3", "x4"]]
y= df["target"]


#split to train & validation  and test sets (80/20)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)

#split the training and validation sets (75/25)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, shuffle=True, random_state=42)

I split the dataset before scaling to avoid data leakage.

## f. Data Preprocessing

### Feature Scaling and Normalization

In [47]:
#Istatnitate the scaler
scaler = MinMaxScaler()

# Fit only on training data, transform all other three sets
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### One hot Encoding

In [48]:
encoder = OneHotEncoder(sparse_output=False)

# Fit on training labels, transform all three
y_train_encoded = encoder.fit_transform(y_train.values.reshape(-1, 1))
y_val_encoded = encoder.transform(y_val.values.reshape(-1, 1))
y_test_encoded = encoder.transform(y_test.values.reshape(-1, 1))

## g. Modeling

### Building the Multilayer Neural Network

In [49]:
#REGRESSOR
mlr = MLPRegressor(hidden_layer_sizes=(20,10), max_iter=100000, activation='relu', random_state=42, verbose=True)

mlr.fit(X_train_scaled, y_train_encoded)

Iteration 1, loss = 0.45010037
Iteration 2, loss = 0.44010923
Iteration 3, loss = 0.43035939
Iteration 4, loss = 0.42082734
Iteration 5, loss = 0.41147469
Iteration 6, loss = 0.40234031
Iteration 7, loss = 0.39342472
Iteration 8, loss = 0.38465318
Iteration 9, loss = 0.37605608
Iteration 10, loss = 0.36765523
Iteration 11, loss = 0.35947872
Iteration 12, loss = 0.35147602
Iteration 13, loss = 0.34361351
Iteration 14, loss = 0.33587497
Iteration 15, loss = 0.32832463
Iteration 16, loss = 0.32091057
Iteration 17, loss = 0.31367506
Iteration 18, loss = 0.30662883
Iteration 19, loss = 0.29971919
Iteration 20, loss = 0.29296146
Iteration 21, loss = 0.28637842
Iteration 22, loss = 0.27992736
Iteration 23, loss = 0.27357728
Iteration 24, loss = 0.26737708
Iteration 25, loss = 0.26134806
Iteration 26, loss = 0.25547392
Iteration 27, loss = 0.24975633
Iteration 28, loss = 0.24417472
Iteration 29, loss = 0.23875011
Iteration 30, loss = 0.23350404
Iteration 31, loss = 0.22841056
Iteration 32, los

Iteration 44, loss = 0.17530233
Iteration 45, loss = 0.17210887
Iteration 46, loss = 0.16902598
Iteration 47, loss = 0.16603226
Iteration 48, loss = 0.16312947
Iteration 49, loss = 0.16031256
Iteration 50, loss = 0.15757820
Iteration 51, loss = 0.15493045
Iteration 52, loss = 0.15236681
Iteration 53, loss = 0.14987788
Iteration 54, loss = 0.14747022
Iteration 55, loss = 0.14514261
Iteration 56, loss = 0.14289375
Iteration 57, loss = 0.14071712
Iteration 58, loss = 0.13860250
Iteration 59, loss = 0.13655688
Iteration 60, loss = 0.13457695
Iteration 61, loss = 0.13266137
Iteration 62, loss = 0.13080315
Iteration 63, loss = 0.12899174
Iteration 64, loss = 0.12723414
Iteration 65, loss = 0.12554576
Iteration 66, loss = 0.12391610
Iteration 67, loss = 0.12232152
Iteration 68, loss = 0.12077046
Iteration 69, loss = 0.11927267
Iteration 70, loss = 0.11781944
Iteration 71, loss = 0.11641445
Iteration 72, loss = 0.11505327
Iteration 73, loss = 0.11375227
Iteration 74, loss = 0.11250226
Iteratio

,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(20, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


### Model validation

In [50]:
#evaluating the model
predictions = mlr.predict(X_val_scaled)

#converting the predictions and the true labels back to their original form
converted_preds = np.argmax(predictions, axis=1)
converted_y_val_encoded = np.argmax(y_val_encoded, axis=1)

#calculating the accuracy
mlr_eval_acc = accuracy_score(converted_y_val_encoded, converted_preds)
print(f"Validation Accuracy: {mlr_eval_acc}")

Validation Accuracy: 0.8333333333333334


## h. Model Testing

In [51]:
#Testing the model
test_predictions = mlr.predict(X_test_scaled)
converted_test_preds = np.argmax(test_predictions, axis=1)
converted_y_test_encoded = np.argmax(y_test_encoded, axis=1)
test_acc = accuracy_score(converted_y_test_encoded, converted_test_preds)
print(f"Test Accuracy: {test_acc}")


Test Accuracy: 0.8
